# Pipeline Centralizado de Extracción y Conversión (CSV -> JSON)

Este cuaderno unifica el procesamiento de `data/amazon.csv` para todos los componentes del dashboard:
1. **KPICardUI**: Total de productos, Rating promedio y Descuento promedio.
2. **BarChartUI**: Conteo de productos por categoría.
3. **DonutChartUI**: Distribución de productos por rangos de calificación.

En cada sección se realiza la transformación, se **visualiza la exportación intermedia a CSV** en `public/data_extracted/` y, al final, se consolidan todos los CSV en **un único archivo JSON** (`public/data_extracted/dashboard_data.json`).

## 1. Configuración, librerías y carga del dataset

In [1]:
import csv
import json
from pathlib import Path

import numpy as np
import pandas as pd

# Detectar la raíz del proyecto sin importar si el cuaderno se ejecuta desde / o /script
cwd = Path.cwd()
project_root = cwd if (cwd / 'data' / 'amazon.csv').exists() else cwd.parent
data_path = project_root / 'data' / 'amazon.csv'
output_dir = project_root / 'public' / 'data_extracted'
output_dir.mkdir(parents=True, exist_ok=True)

print(f'Raíz del proyecto: {project_root}')
print(f'Archivo fuente:    {data_path}')
print(f'Carpeta de salida: {output_dir}')

Raíz del proyecto: c:\Users\Sam\Documents\GitHub\Amazon Sales Dashboard
Archivo fuente:    c:\Users\Sam\Documents\GitHub\Amazon Sales Dashboard\data\amazon.csv
Carpeta de salida: c:\Users\Sam\Documents\GitHub\Amazon Sales Dashboard\public\data_extracted


In [2]:
def export_and_preview_csv(df_or_series, filename: str, preview_lines: int = 8) -> Path:
    """Exporta un DataFrame o Series a CSV y muestra en pantalla el resultado del archivo generado."""
    csv_path = output_dir / filename
    is_series = isinstance(df_or_series, pd.Series)
    df_or_series.to_csv(csv_path, index=is_series, encoding='utf-8')

    lines = csv_path.read_text(encoding='utf-8').splitlines()
    print(f'[CSV exportado] {csv_path.relative_to(project_root)} ({len(lines) - 1} filas de datos)')
    for line in lines[:preview_lines]:
        print(f'  {line}')
    if len(lines) > preview_lines:
        print(f'  ... ({len(lines) - preview_lines} líneas adicionales)')
    return csv_path

df_amazon = pd.read_csv(data_path)
print(f'Dataset cargado: {df_amazon.shape[0]} filas x {df_amazon.shape[1]} columnas')
print('Columnas:', list(df_amazon.columns))
df_amazon[['product_id', 'product_name', 'category', 'discount_percentage', 'rating']].head(5)

Dataset cargado: 1465 filas x 16 columnas
Columnas: ['product_id', 'product_name', 'category', 'discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count', 'about_product', 'user_id', 'user_name', 'review_id', 'review_title', 'review_content', 'img_link', 'product_link']


,product_id,product_name,category,discount_percentage,rating
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,64%,4.2
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,43%,4.0
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,90%,3.9
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,53%,4.2
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...,61%,4.2


## 2. Componente `KPICardUI` — Cálculo y exportación a CSV

Se calculan las tres métricas principales y se exporta cada una a su archivo CSV correspondiente:
- `total_products.csv`
- `average_rating.csv`
- `average_discount.csv`

In [3]:
# 2.1 KPI: Total de productos
total_products = int(len(df_amazon))
total_products_df = pd.DataFrame([{
    'title': 'Total de productos',
    'count': total_products,
}])

total_products_csv = export_and_preview_csv(total_products_df, 'total_products.csv')

[CSV exportado] public\data_extracted\total_products.csv (1 filas de datos)
  title,count
  Total de productos,1465


In [4]:
# 2.2 KPI: Rating promedio de productos
rating_numeric = pd.to_numeric(
    df_amazon['rating'].astype(str).str.replace(',', '.', regex=False),
    errors='coerce',
)
average_rating = float(rating_numeric.mean())
average_rating_df = pd.DataFrame([{
    'title': 'Rating promedio de productos',
    'count': average_rating,
}])

average_rating_csv = export_and_preview_csv(average_rating_df, 'average_rating.csv')

[CSV exportado] public\data_extracted\average_rating.csv (1 filas de datos)
  title,count
  Rating promedio de productos,4.096584699453552


In [5]:
# 2.3 KPI: Descuento promedio
discounts = pd.to_numeric(
    df_amazon['discount_percentage'].astype(str).str.rstrip('%'),
    errors='coerce',
)
average_discount = float(discounts.mean())
average_discount_df = pd.DataFrame([{
    'title': 'Descuento promedio',
    'count': average_discount,
}])

average_discount_csv = export_and_preview_csv(average_discount_df, 'average_discount.csv')

[CSV exportado] public\data_extracted\average_discount.csv (1 filas de datos)
  title,count
  Descuento promedio,47.69146757679181


## 3. Componente `barChartUI` — Productos por categoría

Las categorías vienen concatenadas por `|`. Se separan con `explode()`, se contabilizan y ordenan de mayor a menor antes de exportar a `products_per_category_barChart.csv`.

In [6]:
productos_por_categoria = (
    df_amazon['category']
    .dropna()
    .str.split('|')
    .explode()
    .value_counts()
    .sort_values(ascending=False)
)
productos_por_categoria.index.name = 'category'
productos_por_categoria.name = 'count'

print('Top 6 categorías:')
print(productos_por_categoria.head(6))
print()
bar_chart_csv = export_and_preview_csv(
    productos_por_categoria,
    'products_per_category_barChart.csv',
    preview_lines=10,
)

Top 6 categorías:
category
Electronics                526
Computers&Accessories      453
Home&Kitchen               448
Accessories&Peripherals    381
Kitchen&HomeAppliances     308
Cables                     267
Name: count, dtype: int64

[CSV exportado] public\data_extracted\products_per_category_barChart.csv (317 filas de datos)
  category,count
  Electronics,526
  Computers&Accessories,453
  Home&Kitchen,448
  Accessories&Peripherals,381
  Kitchen&HomeAppliances,308
  Cables,267
  Cables&Accessories,240
  USBCables,233
  SmallKitchenAppliances,181
  ... (308 líneas adicionales)


## 4. Componente `donutChartUI` — Distribución de ratings por rango

Se agrupan las calificaciones en los intervalos `[2, 3)`, `[3, 4)`, `[4, 5)` y `[5, 5.01)` y se exporta el resumen a `rango_ratings.csv`.

In [7]:
limites = [2, 3, 4, 5, 5.01]
etiquetas = ['2 - 2.99', '3 - 3.99', '4 - 4.99', '5']

rango_series = pd.cut(
    rating_numeric,
    bins=limites,
    labels=etiquetas,
    right=False,
)
rango_rating = rango_series.value_counts().sort_index()
rango_rating.index.name = 'rango_rating'
rango_rating.name = 'count'

print('Distribución por rango de rating:')
print(rango_rating)
print()
donut_chart_csv = export_and_preview_csv(rango_rating, 'rango_ratings.csv', preview_lines=10)

Distribución por rango de rating:
rango_rating
2 - 2.99       6
3 - 3.99     348
4 - 4.99    1107
5              3
Name: count, dtype: int64

[CSV exportado] public\data_extracted\rango_ratings.csv (4 filas de datos)
  rango_rating,count
  2 - 2.99,6
  3 - 3.99,348
  4 - 4.99,1107
  5,3


## 5. Consolidación final — Generación de un único JSON compartido

Se leen todos los archivos CSV generados en `public/data_extracted/` y se construye **un único JSON** (`public/data_extracted/dashboard_data.json`) estructurado por componente para que cada vista alcance únicamente su sección correspondiente:
- `kpis.totalProducts`
- `kpis.averageRating`
- `kpis.averageDiscount`
- `barChart`
- `donutChart`

In [8]:
def read_csv_records(csv_path: Path, int_fields: tuple[str, ...] = (), float_fields: tuple[str, ...] = ()) -> list[dict]:
    with csv_path.open('r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    for row in rows:
        for field in int_fields:
            if field in row and row[field] != '':
                row[field] = int(float(row[field]))
        for field in float_fields:
            if field in row and row[field] != '':
                row[field] = float(row[field])
    return rows

dashboard_data = {
    'kpis': {
        'totalProducts': read_csv_records(total_products_csv, int_fields=('count',))[0],
        'averageRating': read_csv_records(average_rating_csv, float_fields=('count',))[0],
        'averageDiscount': read_csv_records(average_discount_csv, float_fields=('count',))[0],
    },
    'barChart': read_csv_records(bar_chart_csv, int_fields=('count',)),
    'donutChart': read_csv_records(donut_chart_csv, int_fields=('count',)),
}

unified_json_path = output_dir / 'dashboard_data.json'
with unified_json_path.open('w', encoding='utf-8') as json_file:
    json.dump(dashboard_data, json_file, indent=4, ensure_ascii=False)

print(f'[JSON único generado] {unified_json_path.relative_to(project_root)} ({unified_json_path.stat().st_size} bytes)')
print('Resumen de secciones en dashboard_data.json:')
print(f"  - kpis.totalProducts:   {dashboard_data['kpis']['totalProducts']}")
print(f"  - kpis.averageRating:   {dashboard_data['kpis']['averageRating']}")
print(f"  - kpis.averageDiscount: {dashboard_data['kpis']['averageDiscount']}")
print(f"  - barChart:             {len(dashboard_data['barChart'])} categorías (top 3: {dashboard_data['barChart'][:3]})")
print(f"  - donutChart:           {len(dashboard_data['donutChart'])} rangos -> {dashboard_data['donutChart']}")

[JSON único generado] public\data_extracted\dashboard_data.json (29819 bytes)
Resumen de secciones en dashboard_data.json:
  - kpis.totalProducts:   {'title': 'Total de productos', 'count': 1465}
  - kpis.averageRating:   {'title': 'Rating promedio de productos', 'count': 4.096584699453552}
  - kpis.averageDiscount: {'title': 'Descuento promedio', 'count': 47.69146757679181}
  - barChart:             317 categorías (top 3: [{'category': 'Electronics', 'count': 526}, {'category': 'Computers&Accessories', 'count': 453}, {'category': 'Home&Kitchen', 'count': 448}])
  - donutChart:           4 rangos -> [{'rango_rating': '2 - 2.99', 'count': 6}, {'rango_rating': '3 - 3.99', 'count': 348}, {'rango_rating': '4 - 4.99', 'count': 1107}, {'rango_rating': '5', 'count': 3}]
